In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PREPROCESSED_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap\training_set_rel3_preprocessed.csv"
)
PROMPT4_TRAITS_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-4.csv"
)

OUTPUT_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-4-concat.csv"
)


In [3]:
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {path.suffix}")

In [5]:
pre = read_table(PREPROCESSED_PATH)
p4_traits = read_table(PROMPT4_TRAITS_PATH)

print("Preprocessed columns:")
print(pre.columns.tolist())

print("\nPrompt 4 trait columns:")
print(p4_traits.columns.tolist())

Preprocessed columns:
['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']

Prompt 4 trait columns:
['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']


In [6]:
pre = pre.rename(columns={
    "Essay ID": "essay_id",
    "essay_id": "essay_id",
    "Essay Set": "essay_set",
    "essay_set": "essay_set",
    "Essay": "essay",
    "essay": "essay",
})

p4_traits = p4_traits.rename(columns={
    "Essay ID": "essay_id",
    "Content": "p4_content",
    "Prompt Adherence": "p4_prompt_adherence",
    "Language": "p4_language",
    "Narrativity": "p4_narrativity",
})

In [7]:
required_pre_cols = [
    "essay_id",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm",
]

trait_cols = [
    "p4_content",
    "p4_prompt_adherence",
    "p4_language",
    "p4_narrativity",
]

required_trait_cols = ["essay_id"] + trait_cols

missing_pre = [c for c in required_pre_cols if c not in pre.columns]
missing_traits = [c for c in required_trait_cols if c not in p4_traits.columns]

if missing_pre:
    raise ValueError(f"Missing columns in preprocessed file: {missing_pre}")

if missing_traits:
    raise ValueError(
        f"Missing columns in prompt 4 trait file: {missing_traits}\n"
        f"Available columns after rename: {p4_traits.columns.tolist()}"
    )


In [8]:
pre_p4 = pre.loc[pre["essay_set"] == 4, required_pre_cols].copy()

pre_p4["essay_id"] = pre_p4["essay_id"].astype(int)
p4_traits["essay_id"] = p4_traits["essay_id"].astype(int)

p4_traits = p4_traits[required_trait_cols].copy()

print("\nRows in preprocessed prompt 4:", len(pre_p4))
print("Rows in prompt 4 trait file:", len(p4_traits))



Rows in preprocessed prompt 4: 1770
Rows in prompt 4 trait file: 1772


In [9]:
dup_traits = p4_traits[
    p4_traits["essay_id"].duplicated(keep=False)
].sort_values("essay_id")

print("\nSố dòng bị trùng essay_id trong p4_traits:", len(dup_traits))
print("Số essay_id bị trùng:", dup_traits["essay_id"].nunique())

if len(dup_traits) > 0:
    display(dup_traits.head(20))


Số dòng bị trùng essay_id trong p4_traits: 0
Số essay_id bị trùng: 0


In [10]:
p4_traits = p4_traits.drop_duplicates()

if p4_traits["essay_id"].duplicated().any():
    print("\nWarning: p4_traits has duplicated essay_id. Aggregating by mean.")
    p4_traits = (
        p4_traits
        .groupby("essay_id", as_index=False)[trait_cols]
        .mean()
    )

# Round half-up nếu có nhiều annotation
for col in trait_cols:
    p4_traits[col] = np.floor(p4_traits[col] + 0.5).astype(int)

print("Duplicate essay_id còn lại:", p4_traits["essay_id"].duplicated().sum())

Duplicate essay_id còn lại: 0


In [11]:
benchmark_p4 = pre_p4.merge(
    p4_traits,
    on="essay_id",
    how="inner",
    validate="one_to_one"
)

benchmark_p4 = benchmark_p4.dropna(subset=trait_cols).copy()

benchmark_p4.insert(
    loc=2,
    column="prompt_id",
    value=4
)

In [12]:
benchmark_p4 = benchmark_p4[
    [
        "essay_id",
        "essay_set",
        "prompt_id",
        "essay",
        "domain1_score",
        "domain1_score_norm",
        "p4_content",
        "p4_prompt_adherence",
        "p4_language",
        "p4_narrativity",
    ]
]


In [13]:

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
benchmark_p4.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("\nDone.")
print(f"Prompt 4 rows in preprocessed: {len(pre_p4)}")
print(f"Prompt 4 rows after merge/dropna: {len(benchmark_p4)}")
print(f"Saved to: {OUTPUT_PATH}")

display(benchmark_p4.head())


Done.
Prompt 4 rows in preprocessed: 1770
Prompt 4 rows after merge/dropna: 1770
Saved to: C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-4-concat.csv


,essay_id,essay_set,prompt_id,essay,domain1_score,domain1_score_norm,p4_content,p4_prompt_adherence,p4_language,p4_narrativity
0,8863,4,4,The author concludes the story with this becau...,0.0,0.00,0,0,0,0
1,8864,4,4,The narrater has that in with Paragraph becuse...,0.0,0.00,0,0,0,0
2,8865,4,4,The author concludes the story with that passa...,3.0,1.00,3,3,3,3
3,8866,4,4,The author ended the story with this paragraph...,2.0,0.67,3,3,2,2
4,8867,4,4,The author concludes the story with this parag...,2.0,0.67,2,2,2,2
